**Notebook Feito por:** MSc. Eng. Paulo de Souza Silva  
**Data:** Agosto de 2026  
**Conteúdo retirado e adaptado de livros e artigos sobre Galerkin Descontínuo**  
**Agradecimentos:** Um agradecimento ao Prof. Dr. Alberto Nogueira pela disponibilização dos scripts em Python para DG 1D

# **Aula 08 - Fluxos Numéricos**

## **Introdução aos Fluxos Numéricos**

Na aula anterior, desvendamos o funcionamento do **termo de volume** da formulação do Método de Galerkin Descontínuo (DG). Vimos como realizar a **projeção dos fluxos físicos** no espaço polinomial e como essa projeção, em conjunto com a Matriz de Rigidez ($\mathcal{S}$), descreve corretamente toda a dinâmica que acontece **no interior de cada elemento** da malha.

Entretanto, ainda existe uma peça importante que falta para completarmos o método.

Para enxergarmos isso com clareza, vamos retomar a formulação fraca (semi-discreta) da equação que estamos resolvendo:

$$
\underbrace{J_e \left[\int_{-1}^{1} \phi_i \phi_j d \xi \right]}_{\text{Jacobiano e Matriz de Massa} \ \mathcal{M}}
\dfrac{\partial}{\partial t} \begin{Bmatrix} c_0 \\ c_1 \\ \vdots \\ c_P \end{Bmatrix} =
\underbrace{\int_{-1}^{1} f \dfrac{\partial}{\partial \xi} \begin{Bmatrix} \phi_0 \\ \phi_1 \\ \vdots \\ \phi_P \end{Bmatrix} d \xi }_{\text{Fluxo, Matriz de Derivação} \ \mathcal{D} \ \text{e Rigidez} \ \mathcal{S}} -
\underbrace{\tilde{f}_{e,e+1} \begin{Bmatrix} \phi^+_0 \\ \phi^+_1 \\ \vdots \\ \phi^+_P \end{Bmatrix} +
\tilde{f}_{e-1,e}\begin{Bmatrix} \phi^-_0 \\ \phi^-_1 \\ \vdots \\ \phi^-_P \end{Bmatrix}}_{\text{Fluxos nas Fronteiras}}
$$

Observe cada um dos blocos da equação.

O lado esquerdo controla a evolução temporal da solução por meio da Matriz de Massa ($\mathcal{M}$). O primeiro termo do lado direito representa toda a dinâmica que ocorre **dentro do elemento**, sendo justamente o operador que estudamos na aula anterior.

Mas e os dois últimos termos destacados?

É exatamente aqui que está um dos conceitos mais importantes do Método de Galerkin Descontínuo.

Como o próprio nome do método sugere, os elementos da malha são **descontínuos**. Em outras palavras, dois elementos vizinhos **não compartilham necessariamente o mesmo valor na interface**. Cada elemento possui sua própria aproximação polinomial e, consequentemente, podem existir dois valores diferentes exatamente na mesma posição física da fronteira.

Isso nos leva naturalmente à seguinte pergunta:

> **Se existem dois valores diferentes na interface, qual deles deve ser utilizado para calcular o fluxo que atravessa essa fronteira?**

Essa pergunta pode parecer simples, mas ela está no coração dos métodos conservativos para Equações Diferenciais Parciais. A resposta não é imediata e, durante décadas, motivou o desenvolvimento de diversas estratégias numéricas capazes de determinar corretamente o fluxo entre duas regiões vizinhas.

No Método de Galerkin Descontínuo, essas estratégias aparecem através dos termos

$$\tilde{f}_{e,e+1} \qquad\text{e}\qquad \tilde{f}_{e-1,e},$$

conhecidos como **Fluxos Numéricos**.

Podemos interpretar esses fluxos como um **árbitro matemático**. Eles observam os dois estados existentes na interface (um vindo do elemento à esquerda e outro do elemento à direita) e determinam um **único fluxo** que será utilizado por ambos os elementos.

Em outras palavras, os Fluxos Numéricos representam o **último componente necessário** para construirmos completamente o operador espacial do Método de Galerkin Descontínuo, denotado por $L_h,$ que posteriormente será integrado no tempo por métodos como Runge-Kutta.

### **Por que estudar Fluxos Numéricos?**
Embora os Fluxos Numéricos sejam fundamentais no DG, eles **não surgiram originalmente nesse método**. 

Na realidade, a necessidade de definir corretamente o fluxo entre regiões vizinhas apareceu muitos anos antes, principalmente em métodos de **Diferenças Finitas** e **Volumes Finitos**, onde também era necessário determinar como a informação deveria atravessar as interfaces entre células.

Por esse motivo, antes de estudarmos os Fluxos Numéricos dentro do contexto do DG, faremos uma breve viagem até esses métodos clássicos. Essa abordagem nos permitirá compreender **por que os Fluxos Numéricos surgiram**, quais problemas eles resolvem e qual é a motivação física por trás dos chamados **Solvers de Riemann**.

Depois dessa motivação, voltaremos naturalmente ao Método de Galerkin Descontínuo e veremos como essas mesmas ideias são incorporadas na formulação do DG.


## **De onde surgiram os Fluxos Numéricos?**

O exemplo mais clássico para compreender a origem dos Fluxos Numéricos é a equação de convecção linear unidimensional. Apesar de extremamente simples, ela possui todas as características fundamentais de problemas hiperbólicos: a informação propaga-se na forma de ondas ao longo do domínio. Isso faz dela um excelente laboratório para entendermos como diferentes discretizações influenciam a estabilidade e a precisão da solução.

Considere a equação de convecção linear 1D, que descreve a propagação de uma onda ao longo do eixo-x com velocidade $a$
$$\frac{\partial u}{\partial t}+a\frac{\partial u}{\partial x}=0$$

Se $a$ for positivo, a solução de onda progressiva da equação acima propaga-se para a direita; o lado esquerdo é denominado lado *upwind* (a montante do fluxo) e o lado direito é o lado *downwind* (a jusante do fluxo). De modo análogo, se $a$ for negativo, a solução de onda progressiva propaga-se para a esquerda; o lado esquerdo é denominado lado *downwind* e o lado direito é o lado *upwind*. 

Se a aproximação da derivada espacial for construída utilizando preferencialmente informações provenientes do lado *upwind* (de onde a informação física está chegando), o esquema é denominado esquema Upwind.


### **Esquema Upwind de Primeira Ordem** 

Vimos no curso da professora Lorena Barba (ver aula 02_01 1D Convection) que um problema desse tipo pode ser escrito de forma discretizada com a estratégia *forward* no tempo e *backward* no espaço quando $a > 0$ 

$$\frac{u_i^{n+1}-u_i^n}{\Delta t} + a \frac{u_i^n - u_{i-1}^n}{\Delta x} = 0$$

e caso $a < 0 $ devemos adotar *forward* no espaço também
$$\frac{u_i^{n+1}-u_i^n}{\Delta t} + a \frac{u_{i+1}^n - u_{i}^n}{\Delta x} = 0$$

Vamos considerar que $a$ é positivo e reorganizarmos, o que nos leva:
$$u_i^{n+1} = u_i^n - a \frac{\Delta t}{\Delta x}(u_i^n-u_{i-1}^n)$$

Vimos ainda que essa técnica é estável se respeitar o coecifiente CFL. Em outras palavras, a informação não pode percorrer mais do que uma célula durante um único passo de tempo. Quando essa condição é violada, o método deixa de representar corretamente a propagação da informação física e a solução torna-se instável.  

$$c = \left|\frac{a \Delta t}{\Delta x} \right| \leq 1$$

#### **Exemplo de Aplicação**

Condição Inicial:
$$u_0(x) = \exp(-0.5(x/0.4)^2)$$

Condição de contorno:
$$u(-2,t) = u(2,t)$$

Note que $a = 0.8$

> Referência: https://www.psvolpiani.com/courses

In [41]:
from IPython.display import HTML, display

display(HTML("""
<div style="display:flex; gap:10px;">
    <img src="animations/advection_Nx101_CFL0.80_T5.0/simulation.gif" width="400">
    <img src="animations/advection_Nx101_CFL1.00_T5.0/simulation.gif" width="400">
    <img src="animations/advection_Nx101_CFL1.20_T5.0/simulation.gif" width="400">
</div>
"""))

#### **O Comportamento da Solução e o Limite CFL**

Como podemos observar nas animações acima, o comportamento do esquema *Upwind* de primeira ordem é extremamente sensível à escolha do passo de tempo $\Delta t$ e do espaçamento da malha $\Delta x$. Analisando os três cenários:

1. **$c = 0.8$ (Dissipação Numérica):** A simulação permanece estável, porém a onda perde amplitude e torna-se progressivamente mais "achatada". Embora o critério CFL seja satisfeito, o erro de truncamento do esquema introduz uma difusão (ou viscosidade) artificial, fazendo com que a solução seja suavizada ao longo do tempo.

2. **$c = 1.0$ (Cenário Ideal):** A onda desloca-se perfeitamente para a direita, preservando sua forma e amplitude. Nesse caso, a informação percorre exatamente um nó da malha a cada passo de tempo, de modo que cada valor é simplesmente transferido para o nó vizinho, sem introduzir dissipação numérica.

3. **$c = 1.2$ (Instabilidade):** A solução diverge rapidamente. Do ponto de vista numérico, a informação física propaga-se mais rapidamente do que a malha consegue transmiti-la. Como consequência, o domínio de dependência numérico deixa de conter o domínio de dependência físico, violando o critério de estabilidade de Courant-Friedrichs-Lewy (CFL).

#### **Além do Esquema Upwind**

Embora o esquema *Upwind* consiga ser estável quando o critério CFL é respeitado, sua elevada dissipação numérica reduz significativamente a precisão da solução. Surge então uma pergunta natural:

> **É possível reduzir essa dissipação sem comprometer a estabilidade do método?**

Uma primeira ideia consiste em abandonar a aproximação unilateral da derivada espacial e utilizar uma diferença central, que possui maior ordem de precisão:

$$\frac{u_i^{n+1}-u_i^n}{\Delta t}+a\frac{u_{i+1}^n-u_{i-1}^n}{2\Delta x}=0$$

Esse é o conhecido esquema **FTCS** (*Forward in Time, Central in Space*). 

À primeira vista, ele parece uma escolha mais precisa do que o esquema *Upwind*. Entretanto, a análise de von Neumann demonstra que, para problemas puramente convectivos, o FTCS é **incondicionalmente instável**, independentemente do valor de $\Delta t$.

Diante dessa limitação, diversos pesquisadores passaram a desenvolver novas discretizações capazes de combinar estabilidade e precisão. 

### **Esquema de Lax-Friedrichs**

Entre as primeiras propostas destaca-se o Esquema de Lax-Friedrichs que modifica o FTCS substituindo o termo $u_i^n$ da derivada temporal pela média dos estados vizinhos, 

$$u_i^n \;\longrightarrow\; \frac{1}{2}(u_{i-1}^n+u_{i+1}^n)$$

substituindo na FTCS
$$\frac{u_i^{n+1}-\frac{1}{2}(u_{i-1}^n+u_{i+1}^n)}{\Delta t}+a\frac{u_{i+1}^n-u_{i-1}^n}{2\Delta x}=0$$

reescrevendo para isolar $u_i^{n+1}$

$$u_i^{n+1}= \frac{1}{2}(u_{i-1}^n+u_{i+1}^n) - a\frac{\Delta t}{2\Delta x} (u_{i+1}^n-u_{i-1}^n)


In [42]:
display(HTML("""
<div style="display:flex; gap:10px;">
    <img src="animations/advection_Nx101_CFL0.80_T5.0/simulation.gif" width="400">
    <img src="LFF_advection_Nx101_CFL0.80_T5.0/simulation.gif" width="400">
</div>
"""))

Comparando os dois resultados, percebe-se que o esquema de Lax-Friedrichs apresenta uma dissipação numérica ainda maior que o esquema Upwind. A maior estabilidade é obtida às custas de uma suavização mais intensa da solução, fazendo com que o pico da onda perca amplitude mais rapidamente.

In [43]:
display(HTML("""
<div style="display:flex; gap:10px;">
    <img src="animations/advection_Nx101_CFL1.20_T5.0/simulation.gif" width="400">
    <img src="LFF_advection_Nx101_CFL1.20_T5.0/simulation.gif" width="400">
</div>
"""))

Embora o esquema de Lax-Friedrichs possua uma dissipação numérica significativamente maior que o Upwind, ambos permanecem sujeitos ao critério de estabilidade CFL. Quando $CFL>1$, a dissipação adicional pode retardar visualmente o crescimento das oscilações, mas não é suficiente para impedir que a solução se torne instável.

### **Esquema de Lax-Wendroff:**

Outra proposta pioneira para reduzir a dissipação do esquema *Upwind* é o **Esquema de Lax-Wendroff**. Diferentemente do Lax-Friedrichs, que estabiliza o FTCS adicionando dissipação artificial, o objetivo do Lax-Wendroff é aumentar a precisão da discretização, obtendo um método de **segunda ordem** no tempo e no espaço.

A ideia central consiste em expandir a solução em Série de Taylor até segunda ordem no tempo,

$$
u_i^{n+1}=u_i^n+\Delta t\frac{\partial u}{\partial t}+\frac{\Delta t^2}{2}\frac{\partial^2u}{\partial t^2},
$$

e utilizar a própria equação da advecção

$$
\frac{\partial u}{\partial t}+a\frac{\partial u}{\partial x}=0
$$

para substituir as derivadas temporais por derivadas espaciais. Após discretizar as derivadas resultantes por diferenças finitas, obtém-se o esquema de Lax-Wendroff:

$$
u_i^{n+1}=u_i^n-\frac{c}{2}(u_{i+1}^n-u_{i-1}^n)+\frac{c^2}{2}(u_{i+1}^n-2u_i^n+u_{i-1}^n),
$$

onde

$$
c=\frac{a\Delta t}{\Delta x}.
$$

Esse esquema continua obedecendo ao critério de estabilidade CFL,

$$
|c|\le1,
$$

porém apresenta um comportamento bastante diferente dos esquemas anteriores.

---

**[Inserir GIFs comparando Upwind × Lax-Friedrichs × Lax-Wendroff para CFL = 0.8]**

Comparando os três resultados, observa-se que o esquema de Lax-Wendroff preserva muito melhor a amplitude e o formato da onda. Enquanto o esquema *Upwind* apresenta difusão numérica e o Lax-Friedrichs suaviza ainda mais a solução, o Lax-Wendroff reduz significativamente essa dissipação graças ao seu erro de truncamento de segunda ordem.

Essa melhora, entretanto, possui um custo. Em regiões onde a solução apresenta gradientes muito elevados ou descontinuidades, o esquema pode desenvolver pequenas oscilações próximas à frente de onda. Esse comportamento é conhecido como **erro dispersivo** e constitui uma das principais limitações dos métodos de alta ordem para problemas convectivos.

---



**[Inserir GIFs comparando Upwind × Lax-Friedrichs × Lax-Wendroff para CFL = 1.2]**

Assim como os demais esquemas explícitos estudados até aqui, o Lax-Wendroff também está sujeito ao critério CFL. Quando $|c|>1$, a solução torna-se instável e os erros crescem rapidamente ao longo das iterações. Embora seu comportamento inicial possa diferir do observado nos esquemas Upwind e Lax-Friedrichs, a violação da condição CFL inevitavelmente conduz à divergência da solução.

Os esquemas Upwind, Lax-Friedrichs e Lax-Wendroff ilustram um importante compromisso presente em praticamente todos os métodos para equações hiperbólicas: aumentar a dissipação melhora a estabilidade, enquanto reduzir a dissipação aumenta a precisão, mas pode introduzir dispersão e oscilações espúrias. Encontrar um equilíbrio entre esses dois efeitos motivou o desenvolvimento de métodos mais sofisticados, culminando nos modernos **Fluxos Numéricos** e **Solvers Aproximados de Riemann**, que estudaremos a seguir no contexto do Método de Galerkin Descontínuo.

### **Como escolher o Fluxo Numérico correto?**

A escolha do Fluxo Numérico depende diretamente da **natureza física da equação** que estamos resolvendo.

- **Problemas Convectivos:** a informação propaga-se em direções bem definidas, como acontece na equação da advecção, em escoamentos invíscidos ou no tráfego de veículos. Nesses casos, o Fluxo Numérico precisa identificar corretamente a direção de propagação das ondas.

- **Problemas Difusivos:** a informação espalha-se em todas as direções, como ocorre na condução de calor ou em fluidos viscosos. Nesses problemas, o Fluxo Numérico deve garantir estabilidade e consistência na aproximação das derivadas espaciais.

Ao longo dos anos, diversas estratégias foram propostas para tratar cada uma dessas situações. A tabela abaixo apresenta alguns dos Fluxos Numéricos mais importantes e que serão estudados ao longo do curso.


| Nome do Fluxo Numérico | Natureza Física | Ideia Principal |
|:---|:---:|:---|
| Upwind (Godunov) | Convectivo | Utiliza apenas a informação proveniente da direção de propagação da onda. |
| Lax-Friedrichs (Rusanov) | Convectivo | Calcula uma média entre os estados vizinhos e adiciona dissipação numérica para estabilizar a solução. |
| Roe | Convectivo | Lineariza localmente o problema utilizando uma Jacobiana aproximada na interface. |
| HLL / HLLC | Convectivo | Estima apenas as ondas dominantes que atravessam a interface para construir o fluxo. |
| Interior Penalty (SIPG/NIPG) | Difusivo | Introduz um termo de penalização para manter a estabilidade da solução nas interfaces. |
| Local Discontinuous Galerkin (LDG) | Difusivo | Reescreve a equação de segunda ordem como um sistema de equações de primeira ordem. |
| Bassi-Rebay (BR1/BR2) | Difusivo | Utiliza operadores de *lifting* para reconstruir gradientes nas interfaces antes de calcular o fluxo. |